##Setup

In [1]:
!pip install gymnasium shimmy ale-py
!pip install autorom
!AutoROM -y

AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/adventure.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/air_raid.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/alien.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/amidar.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/assault.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asterix.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asteroids.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/atlantis.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/atlantis2.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/backgammon.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/bank_heist.bin
Inst

In [ ]:
from collections import defaultdict
import numpy as np
import cv2
import gymnasium as gym
import ale_py
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# ──────────────────────────────────────────────
# Go-Explore cell functions (pixel-based)
# ──────────────────────────────────────────────
def cellfn(frame):
    cell = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    cell = cv2.resize(cell, (11, 8), interpolation=cv2.INTER_AREA)
    cell = cell // 32
    return cell

def hashfn(cell):
    return hash(cell.tobytes())

# ──────────────────────────────────────────────
# Go-Explore archive
# ──────────────────────────────────────────────
e1 = 0.001
e2 = 0.00001

class Weights:
    times_chosen           = 0.1
    times_chosen_since_new = 0.0
    times_seen             = 0.3

class Powers:
    times_chosen           = 0.5
    times_chosen_since_new = 0.5
    times_seen             = 0.5

class Cell:
    def __init__(self):
        self.times_chosen           = 0
        self.times_chosen_since_new = 0
        self.times_seen             = 0

    def __setattr__(self, key, value):
        object.__setattr__(self, key, value)
        if key != 'score' and hasattr(self, 'times_seen'):
            self.score = self.cellscore()

    def cntscore(self, a):
        w = getattr(Weights, a)
        p = getattr(Powers, a)
        v = getattr(self, a)
        return w / (v + e1) ** p + e2

    def cellscore(self):
        return (self.cntscore('times_chosen') +
                self.cntscore('times_chosen_since_new') +
                self.cntscore('times_seen') + 1)

    def visit(self):
        self.times_seen += 1
        return self.times_seen == 1

    def choose(self):
        self.times_chosen           += 1
        self.times_chosen_since_new += 1
        return self.ram, self.reward, self.trajectory


# ──────────────────────────────────────────────
# Networks
# ──────────────────────────────────────────────
class Actor(nn.Module):
    def __init__(self, input_shape, n_actions):
        super().__init__()
        self.conv1 = nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)
        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            x = F.relu(self.conv1(dummy))
            x = F.relu(self.conv2(x))
            x = F.relu(self.conv3(x))
            self._feature_size = x.reshape(1, -1).size(1)
        self.fc = nn.Linear(self._feature_size, n_actions)

    def forward(self, x):
        x = x.float() / 255.
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        # Use log_softmax → softmax path avoids the numerical instability of
        # computing softmax directly when logits are large or contain near-zeros.
        logits = self.fc(x.reshape(x.size(0), -1))
        return F.softmax(logits, dim=-1)


class Critic(nn.Module):
    def __init__(self, input_shape):
        super().__init__()
        self.conv1 = nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)
        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            x = F.relu(self.conv1(dummy))
            x = F.relu(self.conv2(x))
            x = F.relu(self.conv3(x))
            self._feature_size = x.reshape(1, -1).size(1)
        self.fc = nn.Linear(self._feature_size, 1)

    def forward(self, x):
        x = x.float() / 255.
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        return self.fc(x.reshape(x.size(0), -1))


# ──────────────────────────────────────────────
# A2C Agent
# ──────────────────────────────────────────────
class A2CAgent:
    def __init__(self, env, gamma=0.99, lr=1e-4, entropy_coef=0.05, grad_clip=0.5):
        self.env               = env
        self.action_space_size = env.action_space.n
        obs                    = env.observation_space.shape
        self.input_shape       = (obs[2], obs[0], obs[1])   # (C, H, W)

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f'Using device: {self.device}')

        self.actor  = Actor(self.input_shape, self.action_space_size).to(self.device)
        self.critic = Critic(self.input_shape).to(self.device)

        self.actor_optimizer  = optim.Adam(self.actor.parameters(),  lr=lr)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr)

        self.gamma        = gamma
        self.entropy_coef = entropy_coef
        self.grad_clip    = grad_clip

    def select_action(self, state):
        """Sample an action from the current policy. Returns (action int, log_prob tensor)."""
        state_t  = torch.from_numpy(state).permute(2, 0, 1).unsqueeze(0).to(self.device)
        probs    = self.actor(state_t)
        dist     = torch.distributions.Categorical(probs)
        action   = dist.sample()
        return action.item(), dist.log_prob(action)

    def compute_returns(self, rewards, dones):
        R       = 0.0
        returns = []
        for i in reversed(range(len(rewards))):
            if dones[i]:
                R = 0.0
            R = rewards[i] + self.gamma * R
            returns.insert(0, R)
        return torch.tensor(returns, dtype=torch.float32).to(self.device)

    def update_model(self, states, actions, rewards, dones, policy_mask):
        """
        policy_mask : list of bool
            True  → action came from the policy → include in actor loss
            False → action was random           → critic only, skip actor loss

        FIX: random-action steps are excluded from the actor loss entirely.
        Previously we were computing log_probs through the network for random
        actions and including them in the gradient — that's both incorrect
        (off-policy) and the source of the nan crash when those computations
        happened on a 1-element batch.
        """
        # ── Minimum batch size guard ─────────────────────────────────────────
        # FIX: advantage normalisation is undefined for batches of size < 2
        # (std() returns nan). A 1-step episode (die on step 1) triggered this,
        # the nan then propagated through loss.backward() and permanently
        # corrupted all network weights — causing the cascade of nan probs.
        if len(states) < 2:
            return 0.0, 0.0, 0.0

        states_t  = torch.stack(
            [torch.from_numpy(s).permute(2, 0, 1) for s in states]
        ).to(self.device)                                              # (T, C, H, W)
        actions_t = torch.tensor(actions, dtype=torch.long).to(self.device)  # (T,)
        mask_t    = torch.tensor(policy_mask, dtype=torch.bool).to(self.device)

        values  = self.critic(states_t).squeeze(-1)                   # (T,)
        returns = self.compute_returns(rewards, dones)                 # (T,)

        # ── Critic — all steps, Huber loss ───────────────────────────────────
        critic_loss = F.huber_loss(values, returns.detach(), delta=1.0)
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), self.grad_clip)
        self.critic_optimizer.step()

        # ── Actor — policy steps only ─────────────────────────────────────────
        if mask_t.sum() < 2:
            # Entire episode was random (still in warmup) — skip actor update.
            # Also guards against the 1-sample std() nan on the policy subset.
            return 0.0, critic_loss.item(), 0.0

        policy_states  = states_t[mask_t]
        policy_actions = actions_t[mask_t]
        policy_returns = returns[mask_t]
        policy_values  = values[mask_t].detach()

        advantages = policy_returns - policy_values
        # FIX: only normalise when std is well-defined (>1 sample, handled above)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        current_probs = self.actor(policy_states)
        dist          = torch.distributions.Categorical(current_probs)
        log_probs     = dist.log_prob(policy_actions)
        entropy       = dist.entropy().mean()

        actor_loss = -(log_probs * advantages).mean() - self.entropy_coef * entropy

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(self.actor.parameters(), self.grad_clip)
        self.actor_optimizer.step()

        return actor_loss.item(), critic_loss.item(), entropy.item()


# ──────────────────────────────────────────────
# State restoration
# ──────────────────────────────────────────────
def restore_env(env, ram):
    """
    Restore ALE state and return the current screen observation.
    Never calls env.reset() which would overwrite the restored state.
    """
    env.unwrapped.ale.restoreState(ram)
    return env.unwrapped.ale.getScreenRGB()   # (210, 160, 3) uint8


# ──────────────────────────────────────────────
# Hyperparameters
# ──────────────────────────────────────────────
WARMUP_ITERS    = 50    # Pure random Go-Explore before A2C training begins
STEPS_PER_ITER  = 100   # Go-Explore steps per iteration
EXPLORE_EPSILON = 0.5   # Probability of random action during Go-Explore steps

archive   = defaultdict(lambda: Cell())
highscore = 0
frames    = 0

env   = gym.make('ALE/MontezumaRevenge-v5', render_mode='rgb_array')
agent = A2CAgent(env)

state, _     = env.reset()
score        = 0
trajectory   = []
iterations   = 0
restore_cell = None

# ──────────────────────────────────────────────
# Main loop
# ──────────────────────────────────────────────
while True:
    found_new_cell = False

    episode_states    = []
    episode_actions   = []
    episode_rewards   = []
    episode_dones     = []
    episode_policy_mask = []   # True = policy action, False = random action

    for step in range(STEPS_PER_ITER):
        use_random = iterations < WARMUP_ITERS or random.random() < EXPLORE_EPSILON

        if use_random:
            # FIX: for random actions we simply don't compute a log_prob at
            # collection time. The policy_mask tells update_model to skip these
            # steps in the actor loss, so no network forward pass is needed here.
            action      = env.action_space.sample()
            is_policy   = False
        else:
            action, _   = agent.select_action(state)
            is_policy   = True

        next_state, reward, terminal, truncated, info = env.step(action)

        life_lost = info['lives'] < 6
        done      = terminal or truncated or life_lost

        episode_states.append(state)
        episode_actions.append(action)
        episode_rewards.append(reward)
        episode_dones.append(done)
        episode_policy_mask.append(is_policy)

        score += reward
        trajectory.append(action)
        frames += 1

        if score > highscore:
            highscore = score

        if done:
            score      = 0
            trajectory = []
            state, _   = env.reset()
            break
        else:
            cell_repr   = cellfn(next_state)
            cellhash    = hashfn(cell_repr)
            cell        = archive[cellhash]
            first_visit = cell.visit()

            cell_reward = getattr(cell, 'reward',     -1e9)
            cell_traj   = getattr(cell, 'trajectory', [])
            better  = score > cell_reward
            shorter = score == cell_reward and len(trajectory) < len(cell_traj)

            if first_visit or better or shorter:
                cell.ram        = env.unwrapped.ale.cloneState()
                cell.reward     = score
                cell.trajectory = trajectory.copy()
                cell.times_chosen           = 0
                cell.times_chosen_since_new = 0
                found_new_cell = True

            state = next_state

    # ── A2C update ────────────────────────────────────────────────────────────
    if iterations >= WARMUP_ITERS and len(episode_states) > 0:
        actor_loss, critic_loss, entropy = agent.update_model(
            episode_states, episode_actions, episode_rewards,
            episode_dones, episode_policy_mask
        )
    else:
        actor_loss = critic_loss = entropy = 0.0

    if found_new_cell and restore_cell is not None:
        restore_cell.times_chosen_since_new = 0

    iterations += 1

    # ── Select next cell to restore from ─────────────────────────────────────
    if len(archive) > 0:
        scores      = np.array([c.score for c in archive.values()])
        hashes      = list(archive.keys())
        probs       = scores / scores.sum()
        chosen_hash = np.random.choice(hashes, p=probs)
        restore_cell = archive[chosen_hash]
        ram, score, trajectory = restore_cell.choose()
        state = restore_env(env, ram)
    else:
        state, _ = env.reset()
        score      = 0
        trajectory = []

    print(
        f"Iter: {iterations:5d} | Cells: {len(archive):5d} | "
        f"Frames: {frames:8d} | MaxReward: {highscore:.1f} | "
        f"ALoss: {actor_loss:8.4f} | CLoss: {critic_loss:8.4f} | "
        f"Entropy: {entropy:.4f}"
    )

Streaming output truncated to the last 5000 lines.
Iter:   948 | Cells:   213 | Frames:    37678 | MaxReward: 0.0 | ALoss:  -0.2041 | CLoss:   0.0000 | Entropy: 2.8492
Iter:   949 | Cells:   213 | Frames:    37703 | MaxReward: 0.0 | ALoss:  -0.0858 | CLoss:   0.0000 | Entropy: 2.8490
Iter:   950 | Cells:   213 | Frames:    37725 | MaxReward: 0.0 | ALoss:  -0.1142 | CLoss:   0.0000 | Entropy: 2.8494
Iter:   951 | Cells:   213 | Frames:    37726 | MaxReward: 0.0 | ALoss:   0.0000 | CLoss:   0.0000 | Entropy: 0.0000
Iter:   952 | Cells:   213 | Frames:    37736 | MaxReward: 0.0 | ALoss:  -0.0273 | CLoss:   0.0000 | Entropy: 2.8503
Iter:   953 | Cells:   213 | Frames:    37756 | MaxReward: 0.0 | ALoss:  -0.2814 | CLoss:   0.0000 | Entropy: 2.8510
Iter:   954 | Cells:   213 | Frames:    37766 | MaxReward: 0.0 | ALoss:  -0.1507 | CLoss:   0.0000 | Entropy: 2.8512
Iter:   955 | Cells:   213 | Frames:    37778 | MaxReward: 0.0 | ALoss:  -0.0754 | CLoss:   0.0000 | Entropy: 2.8514
Iter:   956 |